[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/41.%20The%20Terminal%20Concession%20Bidding%20Problem/P41-Tier-5_executed.ipynb)

---

# 41. The Terminal Concession Bidding Problem

## Tier 5: Integrated Digital-Twin System-of-Systems Simulation

### Purpose, inputs, outputs, and scope

This notebook builds a deterministic, scenario-based simulation of a terminal concession bid. It connects a port-authority scoring model, simplified competitor responses, operational projections, financial projections, and risk indicators. The purpose is to trace how specified changes in bid strategy affect several modeled outcomes at once.

The inputs are the authority's scoring weights and political factors, competitor response parameters, market and regulatory indicators, a base bid strategy, and a set of scenario changes. For each scenario, the outputs include the modeled winner, that winner's normalized evaluation score, conditional net present value (NPV), system key performance indicators (KPIs), risk indicators, and visual comparisons.

This is a stylized planning model, not a calibrated digital twin of a real port. It does not ingest live data, forecast demand, model negotiation rounds, or estimate statistically calibrated win probabilities. Competitors are represented by adjustments to the base bidder's strategy rather than by fully independent bid profiles. Qualification thresholds are stored as context but are not applied as eligibility gates. Market growth, shipping preferences, trade-lane dynamics, and several competitor attributes are also not used in the scoring calculations. These limits matter when interpreting the results.

### How the simulation works

For each scenario, the model follows a fixed sequence: derive competitor responses from the base strategy and scenario changes; apply the changes to our bid; score all bid strategies; select the highest-scoring bidder; project operations; calculate NPV if our bidder wins; and summarize system-wide KPIs and risks. This is a deterministic sequence of calculations, not an optimization algorithm. The scenarios are specified in advance, and the model evaluates their consequences rather than searching for a best bid.

A **scenario change** is a proportional adjustment. If a base input is \(x\) and its scenario change is \(\delta\), the adjusted input is \(x(1+\delta)\). For example, `0.44` means a 44% increase relative to that base input. Competitor adjustments also depend on the response rules described in the code.

The authority score is assembled in two stages. First, each bid input is converted to a criterion score:

$$
\begin{aligned}
F_b &= 0.8\,\text{financial commitment}_b, \\
T_b &= \text{technology investment}_b, \\
O_b &= 100\,\frac{\text{throughput commitment}_b}{2}, \\
E_b &= \min\!\left(100,\frac{\text{employment commitment}_b}{4}\right), \\
V_b &= 100\,\frac{\text{environmental investment}_b}{50}, \\
L_b &= \frac{\text{local partnership}_b+\text{community investment}_b}{2}.
\end{aligned}
$$

Here, \(b\) identifies a bidder. The six terms are the model's financial, technical, operational, employment, environmental, and local-benefit criterion scores, respectively. The model does not separately cap these scores, except for the employment score shown above. The input scales are part of this stylized model; they should not be interpreted as universal industry standards.

Second, the criterion scores are multiplied by the authority's weights and summed. The model then applies threshold-based political multipliers and caps the final score:

$$
S_b = \min\!\left(100,\; M_b\left(F_b w_F + T_b w_T + O_b w_O + E_b w_E + V_b w_V + L_b w_L\right)\right),
$$

where \(M_b\) is the product of the applicable multipliers:

$$
M_b =
\begin{cases}
1+0.1p_E, & \text{if } E_b\geq 80,\\
1, & \text{otherwise}
\end{cases}
\times
\begin{cases}
1+0.05p_V, & \text{if } V_b\geq 70,\\
1, & \text{otherwise.}
\end{cases}
$$

The weights \(w_F,w_T,w_O,w_E,w_V,w_L\) are stored in `evaluation_weights`. The political factors \(p_E\) and \(p_V\) are `employment_pressure` and `sustainability_focus`. If both thresholds are reached, both multipliers apply. The code implements the criterion conversions, weighted sum, threshold checks, multipliers, and final cap in `evaluate_port_authority_decision`.

The selected winner is the bidder with the largest score. The reported score fraction is \(S_b/100\), a normalized evaluation score and **not** a probability of winning. If scores tie, Python's `max` selects the first bidder in the insertion order of the bid-strategy dictionary. The reported NPV is conditional on our bidder winning; it is not probability-weighted expected NPV.

### Operational and financial calculations

Technology investment determines an automation level, which in turn affects projected capacity, productivity, service quality, operating cost, and reliability. In the code, projected capacity is based on technology investment; the bid's throughput commitment affects its authority score but does not directly set projected capacity. Projected system KPIs aggregate the projections for our bidder and all modeled competitors.

For a winning bid, annual cash flow is calculated in euros using projected capacity in million TEU per year and operating cost in euros per TEU. The model assumes the same annual cash flow for each of 25 years and discounts it at 8%:

$$
\operatorname{NPV}_{\text{million}} = \frac{1}{10^6}\sum_{t=1}^{25}\frac{R-C-I}{(1+0.08)^t},
$$

where \(R\) is annual revenue in euros, \(C\) is annual operating cost in euros, \(I\) is the modeled annualized technology-investment cost in euros, and \(t\) is the year index. Revenue is calculated at €200 per TEU. The code implements this calculation in `simulate_scenario`. If our bidder does not win, the model records an NPV of zero because it does not model the concession's cash flows for other bidders. Consequently, an NPV change between scenarios can reflect a change in whether our bidder wins as well as a change in its projected cash flow.

### Interpreting scenarios and model limits

Scenario comparisons show how this model's assumptions translate into relative changes in scores, projected operations, financial outcomes, and risk indicators. Risk indicators are rule-based values on the model's scale, not calibrated probabilities. A missing risk indicator in a scenario is displayed as unavailable rather than treated as zero. The strategic summary ranks the supplied scenarios according to their modeled scores; its labels and recommendations do not represent an optimization search or a validated investment recommendation.

The model is deterministic: identical inputs produce identical results. Its calculations grow approximately in proportion to the number of scenarios and the number of bidders evaluated per scenario; each bid score uses a fixed number of criteria, and each winning-bid NPV uses a fixed 25-year sum. The system KPIs aggregate projected measures across our bidder and the modeled competitors, so they should be interpreted as outputs of this particular stylized setup, not as independent forecasts of terminal-system performance.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class PortAuthorityDecision:
    """Weights, thresholds, and factors used by the authority model."""

    evaluation_weights: Dict[str, float]
    qualification_thresholds: Dict[str, float]
    political_factors: Dict[str, float]
    regulatory_constraints: List[str]
    negotiation_dynamics: Dict[str, float]


@dataclass
class CompetitorBehavior:
    """Parameters that shape a competitor's modeled response."""

    name: str
    financial_capacity: float
    risk_tolerance: float
    strategic_priorities: Dict[str, float]
    historical_patterns: Dict[str, float]
    response_elasticity: Dict[str, float]


@dataclass
class MarketConditions:
    """Market indicators retained as scenario context."""

    container_growth_rate: float
    economic_outlook: float
    shipping_line_preferences: Dict[str, float]
    trade_lane_dynamics: Dict[str, float]
    competitive_intensity: float


@dataclass
class OperationalMetrics:
    """Projected operational measures for one bid strategy."""

    throughput_capacity: float
    productivity_index: float
    service_quality: float
    operational_cost: float
    reliability_score: float
    automation_level: float


@dataclass
class RegulatoryEnvironment:
    """Regulatory and political indicators used by the model."""

    political_stability: float
    regulatory_complexity: float
    environmental_requirements: Dict[str, float]
    labor_regulations: Dict[str, float]
    policy_trends: Dict[str, float]


@dataclass
class SimulationState:
    """Container for a simulation snapshot and its component inputs."""

    timestamp: datetime
    bidding_phase: str
    active_bidders: List[str]
    market_conditions: MarketConditions
    competitor_strategies: Dict[str, CompetitorBehavior]
    operational_projections: Dict[str, OperationalMetrics]
    regulatory_assessment: RegulatoryEnvironment
    system_kpis: Dict[str, float] = field(default_factory=dict)


@dataclass
class SimulationResults:
    """Outputs for one deterministic scenario simulation."""

    scenario_name: str
    winner: str
    win_score_fraction: float
    npv_million: float
    system_kpis: Dict[str, float]
    interoperability_insights: List[str]
    risk_assessment: Dict[str, float]
    scenario_comparison: Dict[str, float]
    competitor_responses: Dict[str, Dict[str, float]]

In [ ]:
class ConcessionDigitalTwin:
    """Deterministic, scenario-based model of concession-bid interactions."""

    def __init__(self):
        self.port_authority = None
        self.competitors = {}
        self.market_conditions = None
        self.regulatory_environment = None
        self.simulation_history = []
        self.current_state = None

    def initialize_port_authority(self):
        """Set the authority's scoring weights and contextual parameters."""
        self.port_authority = PortAuthorityDecision(
            evaluation_weights={
                "financial_capacity": 0.25,
                "technical_experience": 0.20,
                "operational_plan": 0.20,
                "employment_commitment": 0.15,
                "environmental_score": 0.12,
                "local_benefits": 0.08,
            },
            qualification_thresholds={
                "financial_capacity": 70,
                "technical_experience": 75,
                "operational_plan": 70,
                "employment_commitment": 250,
                "environmental_score": 65,
            },
            political_factors={
                "employment_pressure": 0.8,
                "sustainability_focus": 0.7,
                "economic_development_priority": 0.9,
            },
            regulatory_constraints=[
                "Environmental compliance mandatory",
                "Local employment minimum 250 jobs",
                "Investment timeline maximum 3 years",
            ],
            negotiation_dynamics={
                "price_sensitivity": 0.6,
                "quality_preference": 0.8,
                "relationship_factor": 0.3,
            },
        )

    def initialize_competitors(self):
        """Set competitor attributes used by the response calculation."""
        self.competitors = {
            "MTC": CompetitorBehavior(
                name="Mediterranean Terminal Company",
                financial_capacity=85,
                risk_tolerance=0.4,
                strategic_priorities={"market_share": 0.3, "profitability": 0.5, "growth": 0.2},
                historical_patterns={"bid_aggressiveness": 0.6, "technical_innovation": 0.7, "local_partnership": 0.8},
                response_elasticity={"price_competition": 0.5, "technology_investment": 0.3, "employment_commitment": 0.7},
            ),
            "GPH": CompetitorBehavior(
                name="Global Port Holdings",
                financial_capacity=95,
                risk_tolerance=0.7,
                strategic_priorities={"market_share": 0.6, "profitability": 0.3, "growth": 0.1},
                historical_patterns={"bid_aggressiveness": 0.9, "technical_innovation": 0.8, "local_partnership": 0.5},
                response_elasticity={"price_competition": 0.8, "technology_investment": 0.6, "employment_commitment": 0.4},
            ),
            "ACS": CompetitorBehavior(
                name="Atlantic Container Services",
                financial_capacity=80,
                risk_tolerance=0.5,
                strategic_priorities={"market_share": 0.4, "profitability": 0.4, "growth": 0.2},
                historical_patterns={"bid_aggressiveness": 0.7, "technical_innovation": 0.9, "local_partnership": 0.6},
                response_elasticity={"price_competition": 0.6, "technology_investment": 0.8, "employment_commitment": 0.5},
            ),
        }

    def initialize_market_conditions(self):
        """Set market context; only economic outlook enters the risk score."""
        self.market_conditions = MarketConditions(
            container_growth_rate=4.5,
            economic_outlook=75,
            shipping_line_preferences={"cost_priority": 0.6, "service_quality": 0.3, "reliability": 0.1},
            trade_lane_dynamics={"asia_europe": 0.4, "transatlantic": 0.3, "intra_europe": 0.3},
            competitive_intensity=70,
        )

    def initialize_regulatory_environment(self):
        """Set regulatory context used in the risk calculation."""
        self.regulatory_environment = RegulatoryEnvironment(
            political_stability=80,
            regulatory_complexity=65,
            environmental_requirements={"emission_standards": 0.8, "noise_regulations": 0.6, "water_quality": 0.7},
            labor_regulations={"minimum_wage": 0.5, "working_hours": 0.4, "safety_standards": 0.9},
            policy_trends={"environmental_focus": 0.8, "digitalization_push": 0.7, "labor_protection": 0.6},
        )

    def simulate_competitor_response(
        self,
        base_bidder_strategy: Dict[str, float],
        scenario_changes: Dict[str, float],
    ) -> Dict[str, Dict[str, float]]:
        """Apply elasticity-based adjustments to copies of the base strategy."""
        competitor_responses = {}

        for comp_name, competitor in self.competitors.items():
            response = base_bidder_strategy.copy()
            for factor, change in scenario_changes.items():
                if factor not in competitor.response_elasticity:
                    continue

                elasticity = competitor.response_elasticity[factor]
                risk_adjustment = 1 + (competitor.risk_tolerance - 0.5) * 0.5
                if factor == "technology_investment" and competitor.strategic_priorities["growth"] > 0.3:
                    priority_adjustment = 1.2
                elif factor == "price_competition" and competitor.strategic_priorities["market_share"] > 0.4:
                    priority_adjustment = 1.3
                else:
                    priority_adjustment = 1.0

                total_adjustment = elasticity * risk_adjustment * priority_adjustment
                response[factor] = base_bidder_strategy.get(factor, 0) * (1 + change * total_adjustment)

            competitor_responses[comp_name] = response

        return competitor_responses

    def evaluate_port_authority_decision(
        self, bid_strategies: Dict[str, Dict[str, float]]
    ) -> Dict[str, float]:
        """Calculate a weighted authority score for each supplied bid."""
        scores = {}
        weights = self.port_authority.evaluation_weights

        for bidder, strategy in bid_strategies.items():
            financial_score = strategy.get("financial_commitment", 0) * 0.8
            tech_score = strategy.get("technology_investment", 0)
            operational_score = strategy.get("throughput_commitment", 0) / 2.0 * 100
            employment_score = min(100, strategy.get("employment_commitment", 0) / 4.0)
            env_score = strategy.get("environmental_investment", 0) / 50 * 100
            local_score = (
                strategy.get("local_partnership", 0)
                + strategy.get("community_investment", 0)
            ) / 2

            total_score = (
                financial_score * weights["financial_capacity"]
                + tech_score * weights["technical_experience"]
                + operational_score * weights["operational_plan"]
                + employment_score * weights["employment_commitment"]
                + env_score * weights["environmental_score"]
                + local_score * weights["local_benefits"]
            )

            political_factors = self.port_authority.political_factors
            if employment_score >= 80:
                total_score *= 1 + political_factors["employment_pressure"] * 0.1
            if env_score >= 70:
                total_score *= 1 + political_factors["sustainability_focus"] * 0.05

            scores[bidder] = min(100, total_score)

        return scores

    def calculate_operational_projections(self, strategy: Dict[str, float]) -> OperationalMetrics:
        """Project operating measures from technology and maintenance inputs."""
        base_capacity = 1.5
        tech_investment = strategy.get("technology_investment", 45)
        automation_level = min(95, 40 + tech_investment / 2)
        productivity_multiplier = 1 + (automation_level - 50) / 100
        throughput_capacity = base_capacity * productivity_multiplier
        productivity_index = 25 * productivity_multiplier
        service_quality = 70 + automation_level * 0.3
        operational_cost = 150 - automation_level * 0.5 + tech_investment / 1000
        reliability_score = 75 + automation_level * 0.2 + strategy.get("maintenance_investment", 0) / 100

        return OperationalMetrics(
            throughput_capacity=throughput_capacity,
            productivity_index=productivity_index,
            service_quality=service_quality,
            operational_cost=operational_cost,
            reliability_score=reliability_score,
            automation_level=automation_level,
        )

    def simulate_scenario(
        self,
        scenario_name: str,
        base_strategy: Dict[str, float],
        scenario_changes: Dict[str, float],
    ) -> SimulationResults:
        """Run the model sequence for one named scenario."""
        print(f"\nSimulating Scenario: {scenario_name}")

        competitor_responses = self.simulate_competitor_response(base_strategy, scenario_changes)
        modified_strategy = base_strategy.copy()
        for factor, change in scenario_changes.items():
            if factor in modified_strategy:
                modified_strategy[factor] *= 1 + change

        all_bid_strategies = {"Our_Bidder": modified_strategy}
        all_bid_strategies.update(competitor_responses)
        evaluation_scores = self.evaluate_port_authority_decision(all_bid_strategies)
        winner = max(evaluation_scores, key=evaluation_scores.get)
        win_score_fraction = evaluation_scores[winner] / 100

        operational_projections = {
            bidder: self.calculate_operational_projections(strategy)
            for bidder, strategy in all_bid_strategies.items()
        }

        npv_million = 0.0
        if winner == "Our_Bidder":
            our_ops = operational_projections["Our_Bidder"]
            annual_throughput_teu = our_ops.throughput_capacity * 1_000_000
            annual_revenue_eur = annual_throughput_teu * 200
            annual_operating_cost_eur = annual_throughput_teu * our_ops.operational_cost
            annual_investment_cost_eur = (
                modified_strategy.get("technology_investment", 0) / 10 * 1_000_000
            )
            annual_cash_flow_eur = (
                annual_revenue_eur
                - annual_operating_cost_eur
                - annual_investment_cost_eur
            )
            discount_rate = 0.08
            years = 25
            npv_eur = sum(
                annual_cash_flow_eur / (1 + discount_rate) ** year
                for year in range(1, years + 1)
            )
            npv_million = npv_eur / 1_000_000

        system_kpis = {
            "total_throughput": sum(
                ops.throughput_capacity for ops in operational_projections.values()
            ),
            "average_productivity": float(np.mean([
                ops.productivity_index for ops in operational_projections.values()
            ])),
            "system_reliability": float(np.mean([
                ops.reliability_score for ops in operational_projections.values()
            ])),
            "total_investment": sum(
                strategy.get("technology_investment", 0)
                for strategy in all_bid_strategies.values()
            ),
            "employment_created": sum(
                strategy.get("employment_commitment", 0)
                for strategy in all_bid_strategies.values()
            ),
        }

        insights = []
        if "technology_investment" in scenario_changes:
            tech_change = scenario_changes["technology_investment"]
            if tech_change > 0:
                insights.append(
                    f"Technology investment changes by {tech_change * 100:.0f}%; "
                    "the model links this input to technical scores and automation."
                )
                insights.append(
                    "Technology risk is modeled only when automation exceeds 80; "
                    "the risk indicators show whether this threshold is reached."
                )
        if "financial_commitment" in scenario_changes:
            fin_change = scenario_changes["financial_commitment"]
            insights.append(
                f"Financial commitment changes by {fin_change * 100:.0f}%; "
                "this input affects the authority's financial score."
            )

        our_ops = operational_projections["Our_Bidder"]
        risk_assessment = {
            "operational_risk": max(0, 100 - our_ops.reliability_score),
            "financial_risk": max(0, modified_strategy.get("financial_commitment", 0) - 90) * 2,
            "technology_risk": max(0, our_ops.automation_level - 80) * 1.5,
            "market_risk": (100 - self.market_conditions.economic_outlook) * 0.5,
            "regulatory_risk": self.regulatory_environment.regulatory_complexity * 0.3,
        }

        return SimulationResults(
            scenario_name=scenario_name,
            winner=winner,
            win_score_fraction=win_score_fraction,
            npv_million=npv_million,
            system_kpis=system_kpis,
            interoperability_insights=insights,
            risk_assessment=risk_assessment,
            scenario_comparison=scenario_changes.copy(),
            competitor_responses=competitor_responses,
        )

In [ ]:
def initialize_digital_twin() -> ConcessionDigitalTwin:
    """Create the model and initialize each component."""
    print("=== Initializing Digital-Twin Simulation ===")
    digital_twin = ConcessionDigitalTwin()
    digital_twin.initialize_port_authority()
    digital_twin.initialize_competitors()
    digital_twin.initialize_market_conditions()
    digital_twin.initialize_regulatory_environment()

    print(f"Port authority criteria: {len(digital_twin.port_authority.evaluation_weights)}")
    print(f"Modeled competitors: {len(digital_twin.competitors)} (MTC, GPH, ACS)")
    print(f"Market growth context: {digital_twin.market_conditions.container_growth_rate}% annually")
    print(f"Political stability context: {digital_twin.regulatory_environment.political_stability}/100")
    return digital_twin


digital_twin = initialize_digital_twin()

In [ ]:
def define_base_strategy_and_scenarios() -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """Define the reference bid and proportional what-if changes."""
    base_strategy = {
        "financial_commitment": 85,
        "technology_investment": 45,
        "throughput_commitment": 1.5,
        "employment_commitment": 280,
        "environmental_investment": 15,
        "local_partnership": 60,
        "community_investment": 5,
        "maintenance_investment": 20,
    }

    scenarios = {
        "Baseline": {},
        "Increased_Technology_Investment": {"technology_investment": 0.44},
        "Aggressive_Financial_Commitment": {"financial_commitment": 0.18},
        "Enhanced_Employment_Commitment": {"employment_commitment": 0.43},
        "Comprehensive_Strategy": {
            "technology_investment": 0.44,
            "financial_commitment": 0.18,
            "employment_commitment": 0.43,
            "environmental_investment": 0.33,
        },
        "Market_Disruption": {
            "technology_investment": 0.78,
            "financial_commitment": 0.35,
        },
    }

    print("\nBase Strategy Defined:")
    for component, value in base_strategy.items():
        label = component.replace("_", " ").title()
        if component == "employment_commitment":
            print(f"- {label}: {value} jobs")
        elif component == "local_partnership":
            print(f"- {label}: {value} points")
        elif component == "throughput_commitment":
            print(f"- {label}: {value} million TEU per year")
        else:
            print(f"- {label}: €{value} million")

    print(f"\nScenarios defined: {len(scenarios)}")
    for scenario_name in scenarios:
        print(f"- {scenario_name.replace('_', ' ')}")
    return base_strategy, scenarios


base_strategy, scenarios = define_base_strategy_and_scenarios()

### Running and reading the scenarios

Each scenario value is a proportional change applied to the relevant base-strategy input. For a base value \(x\) and scenario change `δ`, the adjusted value is \(x(1+\delta)\). For example, a change of `0.44` multiplies the base value by `1 + 0.44`; it is a proportional scenario specification, not an exact target value. Competitor responses use the same proportional change, modified by each competitor's response elasticity, risk tolerance, and the limited priority rules in the response function.

The scenario sequence is the same for each supplied case: adjust competitor strategies, modify our bid, score the bidders, select the winner, calculate operational projections, calculate conditional NPV when applicable, and summarize KPIs and risk indicators. The KPIs aggregate projected measures across our bidder and the modeled competitors. They should therefore be interpreted as outputs of this particular stylized setup, not as independent forecasts of terminal-system performance.

In [ ]:
def run_all_scenarios(
    digital_twin: ConcessionDigitalTwin,
    base_strategy: Dict[str, float],
    scenarios: Dict[str, Dict[str, float]],
) -> Dict[str, SimulationResults]:
    """Run each scenario and print its principal modeled outcomes."""
    print("\n=== Running Scenario Simulations ===")
    all_results = {}

    for scenario_name, scenario_changes in scenarios.items():
        print(f"\n{'=' * 50}")
        result = digital_twin.simulate_scenario(
            scenario_name, base_strategy, scenario_changes
        )
        all_results[scenario_name] = result
        print(f"Simulated winner: {result.winner}")
        print(f"Winner's normalized evaluation score: {result.win_score_fraction:.2%}")
        if result.winner == "Our_Bidder":
            print(f"Conditional NPV: €{result.npv_million:.1f} million")
        print(f"System throughput: {result.system_kpis['total_throughput']:.2f} million TEU/year")
        print(f"Total modeled employment: {result.system_kpis['employment_created']:.0f} jobs")

    return all_results


simulation_results = run_all_scenarios(digital_twin, base_strategy, scenarios)

In [ ]:
def analyze_interoperability_insights(
    all_results: Dict[str, SimulationResults],
) -> None:
    """Compare modeled effects across the baseline and technology scenario."""
    print("\n=== Cross-System Scenario Analysis ===")
    baseline_result = all_results["Baseline"]
    tech_result = all_results["Increased_Technology_Investment"]

    print("\nTechnology Investment Scenario:")
    print(f"Baseline winner's score fraction: {baseline_result.win_score_fraction:.2%}")
    print(f"Technology scenario winner's score fraction: {tech_result.win_score_fraction:.2%}")
    score_change = tech_result.win_score_fraction - baseline_result.win_score_fraction
    print(f"Change in normalized score: {score_change * 100:+.2f} percentage points")

    if baseline_result.winner == "Our_Bidder" and tech_result.winner == "Our_Bidder":
        npv_change = tech_result.npv_million - baseline_result.npv_million
        print(f"Baseline conditional NPV: €{baseline_result.npv_million:.1f} million")
        print(f"Technology scenario conditional NPV: €{tech_result.npv_million:.1f} million")
        print(f"Change in conditional NPV: €{npv_change:+.1f} million")

    print("\nRisk indicator changes (technology scenario minus baseline):")
    for risk_type, baseline_value in baseline_result.risk_assessment.items():
        change = tech_result.risk_assessment[risk_type] - baseline_value
        if np.isclose(change, 0.0):
            direction = "unchanged"
        else:
            direction = "increased" if change > 0 else "decreased"
        print(f"- {risk_type.replace('_', ' ').title()}: {direction} by {abs(change):.2f}")

    print("\nSystem KPI changes (technology scenario minus baseline):")
    for kpi, baseline_value in baseline_result.system_kpis.items():
        change = tech_result.system_kpis[kpi] - baseline_value
        print(f"- {kpi.replace('_', ' ').title()}: {change:+.2f}")

    print("\nModeled scenario insights:")
    for scenario_name, result in all_results.items():
        if result.interoperability_insights:
            print(f"\n{scenario_name.replace('_', ' ')}:")
            for insight in result.interoperability_insights:
                print(f"- {insight}")


analyze_interoperability_insights(simulation_results)

In [ ]:
def visualize_digital_twin_results(
    all_results: Dict[str, SimulationResults],
    base_strategy: Dict[str, float],
    scenarios: Dict[str, Dict[str, float]],
) -> plt.Figure:
    """Plot scenario scores, financial projections, KPIs, risks, and responses."""
    scenario_names = list(all_results)
    results = list(all_results.values())
    labels = [name.replace("_", "\n") for name in scenario_names]
    colors = sns.color_palette("husl", n_colors=len(scenario_names))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("Scenario-Based Concession Simulation", fontsize=16, fontweight="bold")

    scores = [result.win_score_fraction for result in results]
    bars = axes[0, 0].bar(range(len(labels)), scores, color=colors, alpha=0.85)
    for bar, score in zip(bars, scores):
        axes[0, 0].text(
            bar.get_x() + bar.get_width() / 2,
            score + 0.01,
            f"{score:.1%}",
            ha="center",
            va="bottom",
        )
    axes[0, 0].set_title("Winner's Normalized Evaluation Score")
    axes[0, 0].set_ylabel("Score fraction (0–1; not a probability)")
    axes[0, 0].set_xticks(range(len(labels)), labels, rotation=35, ha="right")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(axis="y", alpha=0.3)

    npvs = [
        result.npv_million if result.winner == "Our_Bidder" else 0
        for result in results
    ]
    bars = axes[0, 1].bar(range(len(labels)), npvs, color=colors, alpha=0.85)
    for bar, npv in zip(bars, npvs):
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"€{npv:.0f}M",
            ha="center",
            va="bottom",
        )
    axes[0, 1].set_title("Conditional NPV for Our Bidder")
    axes[0, 1].set_ylabel("NPV (million euros; zero if our bidder loses)")
    axes[0, 1].set_xticks(range(len(labels)), labels, rotation=35, ha="right")
    axes[0, 1].grid(axis="y", alpha=0.3)

    baseline_kpis = all_results["Baseline"].system_kpis
    comprehensive_kpis = all_results["Comprehensive_Strategy"].system_kpis
    kpi_names = ["total_throughput", "average_productivity", "system_reliability"]
    kpi_labels = ["Throughput", "Productivity", "Reliability"]
    divisors = [8.0, 40.0, 100.0]
    baseline_values = [
        baseline_kpis[name] / divisor
        for name, divisor in zip(kpi_names, divisors)
    ]
    comprehensive_values = [
        comprehensive_kpis[name] / divisor
        for name, divisor in zip(kpi_names, divisors)
    ]
    angles = np.linspace(0, 2 * np.pi, len(kpi_names), endpoint=False).tolist()
    angles += angles[:1]
    baseline_values += baseline_values[:1]
    comprehensive_values += comprehensive_values[:1]
    axes[0, 2].plot(angles, baseline_values, "o-", linewidth=2, label="Baseline")
    axes[0, 2].fill(angles, baseline_values, alpha=0.2)
    axes[0, 2].plot(
        angles,
        comprehensive_values,
        "o-",
        linewidth=2,
        label="Comprehensive strategy",
    )
    axes[0, 2].fill(angles, comprehensive_values, alpha=0.2)
    axes[0, 2].set_xticks(angles[:-1], kpi_labels)
    axes[0, 2].set_ylim(0, 1)
    axes[0, 2].set_title("Selected System KPIs (Scaled)")
    axes[0, 2].set_ylabel("Scaled value; KPI-specific scale")
    axes[0, 2].legend()
    axes[0, 2].grid(True)

    risk_scenarios = [
        "Baseline",
        "Increased_Technology_Investment",
        "Comprehensive_Strategy",
    ]
    risk_keys = list(dict.fromkeys(
        key
        for scenario_name in risk_scenarios
        for key in all_results[scenario_name].risk_assessment
    ))
    risk_labels = [key.removesuffix("_risk").title() for key in risk_keys]
    risk_matrix = np.array([
        [all_results[name].risk_assessment.get(key, np.nan) for name in risk_scenarios]
        for key in risk_keys
    ], dtype=float)
    masked_risk_matrix = np.ma.masked_invalid(risk_matrix)
    image = axes[1, 0].imshow(
        masked_risk_matrix,
        cmap="RdYlBu_r",
        aspect="auto",
    )
    axes[1, 0].set_xticks(
        range(len(risk_scenarios)),
        ["Baseline", "Technology", "Comprehensive"],
        rotation=30,
    )
    axes[1, 0].set_yticks(range(len(risk_labels)), risk_labels)
    axes[1, 0].set_title("Modeled Risk Indicators")
    for row in range(risk_matrix.shape[0]):
        for column in range(risk_matrix.shape[1]):
            value = risk_matrix[row, column]
            label = f"{value:.1f}" if np.isfinite(value) else "N/A"
            axes[1, 0].text(
                column,
                row,
                label,
                ha="center",
                va="center",
                color="black",
            )
    fig.colorbar(image, ax=axes[1, 0], label="Risk indicator (model scale)")

    tech_result = all_results["Increased_Technology_Investment"]
    competitor_names = list(tech_result.competitor_responses)
    competitor_investments = [
        tech_result.competitor_responses[name].get("technology_investment", 0)
        for name in competitor_names
    ]
    competitor_names.append("Our bidder")
    our_technology_investment = base_strategy["technology_investment"] * (
        1 + scenarios["Increased_Technology_Investment"]["technology_investment"]
    )
    competitor_investments.append(our_technology_investment)
    bars = axes[1, 1].bar(competitor_names, competitor_investments, alpha=0.85)
    for bar, amount in zip(bars, competitor_investments):
        axes[1, 1].text(
            bar.get_x() + bar.get_width() / 2,
            amount + 1,
            f"€{amount:.1f}M",
            ha="center",
            va="bottom",
        )
    axes[1, 1].set_ylabel("Modeled technology investment (million euros)")
    axes[1, 1].set_title("Strategies in the Technology Scenario")
    axes[1, 1].grid(axis="y", alpha=0.3)

    baseline = all_results["Baseline"]
    table_data = []
    for name, result in all_results.items():
        if name == "Baseline":
            continue
        score_change = (result.win_score_fraction - baseline.win_score_fraction) * 100
        npv_change = result.npv_million - baseline.npv_million
        table_data.append([
            name.replace("_", " "),
            f"{score_change:+.2f} pp",
            f"€{npv_change:+.1f}M",
            result.winner,
        ])
    axes[1, 2].axis("off")
    table = axes[1, 2].table(
        cellText=table_data,
        colLabels=["Scenario", "Score change", "Recorded NPV change", "Winner"],
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.7)
    axes[1, 2].set_title("Scenario Changes Relative to Baseline", pad=18)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_digital_twin_results(
    simulation_results,
    base_strategy,
    scenarios,
)

In [ ]:
def generate_strategic_recommendations(
    all_results: Dict[str, SimulationResults],
) -> Dict[str, Any]:
    """Summarize the strongest modeled scenario and its indicators."""
    print("\n=== Scenario-Based Strategic Summary ===")
    winning_scenarios = [
        item for item in all_results.items() if item[1].winner == "Our_Bidder"
    ]
    if not winning_scenarios:
        best_scenario_name, best_result = max(
            all_results.items(), key=lambda item: item[1].win_score_fraction
        )
    else:
        best_scenario_name, best_result = max(
            winning_scenarios, key=lambda item: item[1].win_score_fraction
        )

    print(f"\nHighest-scoring modeled scenario: {best_scenario_name.replace('_', ' ')}")
    print(f"Winner: {best_result.winner}")
    print(f"Winner's normalized evaluation score: {best_result.win_score_fraction:.2%}")
    if best_result.winner == "Our_Bidder":
        print(f"Conditional NPV: €{best_result.npv_million:.1f} million")

    baseline = all_results["Baseline"]
    success_factors = []
    comparisons = [
        ("Increased_Technology_Investment", "Technology investment", 0.02,
         "Review the modeled technology and automation assumptions."),
        ("Aggressive_Financial_Commitment", "Financial commitment", 0.01,
         "Review the modeled financial-capacity scoring relationship."),
        ("Enhanced_Employment_Commitment", "Employment commitment", 0.015,
         "Review how employment commitments affect the modeled authority score."),
    ]
    for scenario_name, factor_name, threshold, recommendation in comparisons:
        if scenario_name not in all_results:
            continue
        improvement = all_results[scenario_name].win_score_fraction - baseline.win_score_fraction
        if improvement > threshold:
            success_factors.append({
                "factor": factor_name,
                "score_change": improvement,
                "recommendation": recommendation,
            })

    risk_summary = {"low_risk_factors": [], "high_risk_factors": [], "mitigation_strategies": []}
    for risk_type, risk_score in best_result.risk_assessment.items():
        label = risk_type.replace("_", " ").title()
        if risk_score < 20:
            risk_summary["low_risk_factors"].append(label)
        elif risk_score > 40:
            risk_summary["high_risk_factors"].append(label)

    if "Technology Risk" in risk_summary["high_risk_factors"]:
        risk_summary["mitigation_strategies"].append(
            "Assess technology obsolescence assumptions and consider staged investment."
        )
    if "Financial Risk" in risk_summary["high_risk_factors"]:
        risk_summary["mitigation_strategies"].append(
            "Review financing assumptions and the modeled financial-risk threshold."
        )

    integration_insights = list(best_result.interoperability_insights)
    integration_insights.append(
        "Competitor outcomes depend on the response rules and shared base-strategy representation."
    )

    implementation_roadmap = [
        {
            "phase": "Pre-bid preparation",
            "timeline": "Months 1-3",
            "activities": [
                "Validate model inputs against evidence relevant to the competition.",
                "Review financing assumptions and technology plans.",
                "Document uncertainty that the deterministic scenarios do not represent.",
            ],
        },
        {
            "phase": "Bid submission",
            "timeline": "Month 4",
            "activities": [
                "Use scenario results as structured decision support, not as a guaranteed outcome.",
                "Present the bid's operational and risk assumptions transparently.",
                "Check commitments against applicable requirements.",
            ],
        },
        {
            "phase": "Post-selection",
            "timeline": "Months 5-36",
            "activities": [
                "Compare realized operational measures with the projections.",
                "Reassess investment and implementation assumptions as evidence changes.",
                "Track commitments and risk indicators over time.",
            ],
        },
    ]

    recommendations = {
        "selected_scenario": best_scenario_name,
        "expected_outcomes": {
            "winner": best_result.winner,
            "normalized_score_fraction": best_result.win_score_fraction,
            "conditional_npv_million": best_result.npv_million,
            "system_benefits": best_result.system_kpis,
        },
        "success_factors": success_factors,
        "risk_assessment": risk_summary,
        "integration_insights": integration_insights,
        "implementation_roadmap": implementation_roadmap,
    }

    print("\nScenario comparisons with notable score changes:")
    for factor in success_factors:
        print(f"- {factor['factor']}: {factor['score_change'] * 100:+.2f} percentage points")
        print(f"  Interpretation: {factor['recommendation']}")
    print("\nRisk indicators for the selected scenario:")
    for risk_type, risk_score in best_result.risk_assessment.items():
        print(f"- {risk_type.replace('_', ' ').title()}: {risk_score:.2f} (model scale)")
    print("\nImplementation roadmap:")
    for phase in implementation_roadmap:
        print(f"- {phase['phase']} ({phase['timeline']}):")
        for activity in phase["activities"]:
            print(f"  - {activity}")
    return recommendations


recommendations = generate_strategic_recommendations(simulation_results)

In [ ]:
def generate_digital_twin_summary(
    all_results: Dict[str, SimulationResults],
    recommendations: Dict[str, Any],
) -> None:
    """Print a concise account of model structure and scenario outcomes."""
    baseline = all_results["Baseline"]
    selected_name = recommendations["selected_scenario"]
    selected = all_results[selected_name]
    score_change = selected.win_score_fraction - baseline.win_score_fraction

    print("\n" + "=" * 78)
    print("TERMINAL CONCESSION BIDDING: SCENARIO SIMULATION SUMMARY")
    print("=" * 78)
    print("\nModel structure:")
    print("- Components: authority scoring, competitor responses, operations, finance, and risk indicators")
    print("- Method: deterministic, sequential what-if scenario simulation")
    print("- Important limit: normalized score fractions are not calibrated win probabilities")
    print("- Important limit: the model is not connected to live operational or market data")

    print("\nScenario summary:")
    print(f"- Baseline winner: {baseline.winner}")
    print(f"- Baseline winner's normalized score: {baseline.win_score_fraction:.2%}")
    print(f"- Highest-scoring selected scenario: {selected_name.replace('_', ' ')}")
    print(f"- Selected scenario winner: {selected.winner}")
    print(f"- Selected scenario winner's normalized score: {selected.win_score_fraction:.2%}")
    print(f"- Score change relative to baseline: {score_change * 100:+.2f} percentage points")
    if selected.winner == "Our_Bidder":
        print(f"- Conditional NPV: €{selected.npv_million:.1f} million")
    print(f"- Modeled system throughput: {selected.system_kpis['total_throughput']:.2f} million TEU/year")
    print(f"- Modeled total employment: {selected.system_kpis['employment_created']:.0f} jobs")

    print("\nInterpretation notes:")
    print("- The simulated winner is the highest-scoring bidder under the specified scoring rules.")
    print("- Competitor strategies are response-adjusted copies of the base bidder strategy.")
    print("- The NPV projection uses constant annual cash flow over the modeled concession horizon.")
    print("- Scenario results depend on the chosen inputs, scoring scales, and simplifying assumptions.")

    print("\nRoadmap phases:")
    for phase in recommendations["implementation_roadmap"]:
        print(f"- {phase['phase']}: {phase['timeline']}")
    print("\nUse the scenario outputs to inspect modeled relationships and assumptions, not as a substitute for evidence-based bid appraisal.")
    print("=" * 78)


generate_digital_twin_summary(simulation_results, recommendations)